# Session 1. An agent is a loop

**A working agent by the end. No framework in it.**

- a `for` statement, a text parser, one plain Python function
- the course contract: separate document


## The model as a component

**One function. Messages in, one message back.**

- `system`, `user`, `assistant`: the whole API surface today


In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# Reads the .env sitting next to this notebook, or in any directory above it.
load_dotenv()

# No wrapper today: the provider's own client, key and endpoint straight out of
# the environment. Nothing is hidden yet, and that is the point of today.
API_KEY = os.environ["LLM_API_KEY"]
# Providers disagree about the tail of the address: a version segment, a longer
# path, a trailing slash. Most frequent typo in the room, and it comes back as a
# 404 that reads like an authentication problem.
BASE_URL = os.environ["LLM_BASE_URL"]
# Model names never live in code: MODEL_CHEAP here, MODEL_STRONG from session 2.
# Two families named in the pre-course document shut down inside this semester,
# one of them in October, around session 6. Edit .env, not this.
MODEL = os.environ["MODEL_CHEAP"]

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
# One line of proof that .env was found, before anything else runs.
print("endpoint:", BASE_URL, "| model:", MODEL)


In [ ]:
answer = client.chat.completions.create(
    model=MODEL,
    # The entire state of the conversation, sent on this call. There is no
    # session id and no server-side history to refer to.
    messages=[
        # system: standing instructions. user: what you typed.
        {"role": "system", "content": "You are terse. Two sentences at most."},
        {"role": "user", "content": "Name one book about graph algorithms."},
    ],
    # High on purpose: a reasoning model spends this budget before it writes
    # a visible character, and an empty reply then looks like a parser bug.
    max_completion_tokens=2000,
)

# Read the object, not the text. `choices` is a list because you may ask for
# several completions; `message` carries a role, a content, and a `tool_calls`
# field that is empty today and becomes the point of the last section.
print(answer.choices[0].message)
print()
# `stop` = finished the sentence. `length` = ran out of budget mid-word.
print("finish_reason:", answer.choices[0].finish_reason)
# The bill. Completion tokens include the hidden reasoning ones.
print("usage:        ", answer.usage)


In [ ]:
def ask(messages):
    reply = client.chat.completions.create(
        model=MODEL, messages=messages, max_completion_tokens=2000
    )
    # Throw away everything except the text. The full object came back above.
    return reply.choices[0].message.content


Q = "Name one book about graph algorithms."

# Same input twice, so the two answers may match word for word. That is
# determinism, not memory: nothing here refers back to anything.
print("call 1:", ask([{"role": "user", "content": Q}]))
print()
print("call 2:", ask([{"role": "user", "content": Q}]))
print()
# Who wrote *it*? This call carries no trace of the two above.
print("call 3:", ask([{"role": "user", "content": "Who wrote it?"}]))


**Three requests, three strangers.**

- call 3 asks who wrote *it*: there is no *it*, it answers anyway


In [ ]:
# One real call, then we play both sides.
first = ask([{"role": "user", "content": Q}])

# Memory, all semester, is something appending to a list like this one. Memory
# bugs are the wrong list, an overwritten list, or a copy.
conversation = [
    {"role": "user", "content": Q},
    # The assistant turn is ours to append. The provider did not keep it.
    {"role": "assistant", "content": first},
    {"role": "user", "content": "Who wrote it?"},
]

# Same question as call 3. This time *it* has an antecedent in the list.
print(ask(conversation))


**The conversation is a list you own and resend.**

- the provider stores nothing between calls
- sessions 2-16 add machinery, none replaces the list
- a black box with an interface; we teach the interface
- the box itself: https://karpathy.ai/zero-to-hero.html


## Access channels

**Table is in the pre-course document. One rule out loud.**

- free tier: assume the provider keeps what you send and that people read it
- tool output goes there too, so fictional data only


## The loop

**The agent needs something to call.**


In [ ]:
BOOKS = [
    ("Introduction to Algorithms", "algorithms, data structures, graphs"),
    ("The Pragmatic Programmer", "craft, habits, career"),
    ("Designing Data-Intensive Applications", "databases, distributed systems"),
    ("Structure and Interpretation of Computer Programs", "recursion, interpreters"),
    ("Refactoring", "code smells, tests, design"),
    ("The Mythical Man-Month", "teams, estimation, schedules"),
]


# No decorator, no base class. Nothing in here knows a language model exists.
def find_book(query: str) -> str:
    """Search the catalogue by title or topic. One short word works best."""
    needle = query.strip().lower()
    # Substring match on title or topics. Deliberately dumb, so it can miss.
    hits = [t for t, topics in BOOKS if needle in t.lower() or needle in topics]
    if not hits:
        # Written for the model, never for you: it names the fix and gives three
        # words that work. Return None or raise KeyError and the agent gives up.
        return f"No match for {query!r}. Try a single keyword, for example: graphs, tests, databases."
    return "; ".join(hits)


print(find_book("graphs"))
print(find_book("recursion"))
# A whole phrase, no match. This is the line the model hits first.
print(find_book("something about graph algorithms"))


**The failure message is the model's only input to its next move.**

- session 3 is largely about this one sentence


### The system prompt


In [ ]:
# A model produces text and nothing else, so a tool call has to be text too,
# in a shape agreed in advance that `str.startswith` can find again.
SYSTEM = """You answer questions using a book catalogue.

Work in a loop and use exactly this format:

Thought: what you need to find out
Action: find_book("<query>")

Then STOP and wait. The result comes back as:

Observation: <what the tool returned>

Never write an Observation yourself. When you know the answer, reply with:

Final Answer: <your answer>
"""

# A phrase, not a keyword: chosen so the first tool call misses.
QUESTION = "Can you recommend me something about graph algorithms?"

# Read it once out loud. This text is the whole tool-calling protocol.
print(SYSTEM)


### Cutting the reply off

**"Then STOP and wait" is a request, not a mechanism.**

- given the chance, the model writes the Observation itself, then answers


In [ ]:
# The provider-side `stop` parameter is dead for us: 400 on reasoning models,
# and missing from some providers' request schemas. So we cut on our side.
def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    # -1 is the normal case on the final turn: nothing to cut.
    return text if marker == -1 else text[:marker].rstrip()


# What an uncut reply looks like: the model answered its own tool call.
FAKE = """Thought: I should look this up.
Action: find_book("graphs")
Observation: Introduction to Algorithms, third edition.
Final Answer: Read Introduction to Algorithms."""

# Two lines of ours. They work on every endpoint in the pre-course table.
print(cut_at_observation(FAKE))


In [ ]:
def parse_action(text: str) -> str | None:
    """Pull the query out of: Action: find_book("graphs")"""
    for line in text.splitlines():
        line = line.strip()
        # One line begins with `Action:`; take what sits between the brackets.
        # Cleverer parsers exist and fail in more interesting ways.
        if line.startswith("Action:") and "(" in line:
            return line[line.index("(") + 1 : line.rindex(")")].strip().strip("\"'")
    # None means the model is done: the loop's only exit signal. Remember that
    # in the crash-path section, where a truncated reply also parses to None.
    return None


print(repr(parse_action(FAKE)))
# No Action line, so None. That is how a Final Answer reaches the user.
print(repr(parse_action("Final Answer: Introduction to Algorithms.")))


### Handing the result back

**The observation goes back as a `user` message.**

- the API will not continue a trailing assistant message
- Hugging Face appends it to the assistant turn: does not port
- six moves: send, cut, find Action, run, append two, send again
- one of the six belongs to the model. The rest is our code


In [ ]:
def run(honest: bool = True, steps: int = 5) -> list[str]:
    # The two seed messages. Everything the model ever sees starts here.
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": QUESTION},
    ]
    # Collected for the practice artifact: the run log you will commit.
    transcript = [f"Question: {QUESTION}"]

    # The bound is the only thing between a confused model and your quota.
    for _ in range(steps):
        answer = client.chat.completions.create(
            model=MODEL,
            # Resent in full every pass. This list is the only memory there is.
            messages=messages,
            # Generous on purpose. The two cells after the break-it run show why.
            max_completion_tokens=2000,
        )
        # `content` can be None, not "". The `or` is load-bearing.
        reply = answer.choices[0].message.content or ""
        # honest=False skips the cut. That is the break-it run below.
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        # Watch this on screen: one Thought and one Action per pass.
        print(reply)

        query = parse_action(reply)
        # No Action line: the model is finished, or believes it is.
        if query is None:
            break

        # Our Python runs. The only place the catalogue is ever touched.
        observation = f"Observation: {find_book(query)}"
        transcript.append(observation)
        print(observation)

        # Both turns, in order: what the model said, then what we saw.
        messages.append({"role": "assistant", "content": reply})
        # role=user, because the API will not continue an assistant turn.
        messages.append({"role": "user", "content": observation})

    return transcript


**Bound every loop you write this semester.**

- session 2 calls it `recursion_limit`, default 10007 not 25


In [ ]:
# Count the Action lines as they print. There should be two.
transcript = run()


**Two Action lines. The second one is the whole course.**

- the miss came back with advice, the model narrowed to one word, then answered
- its output changed what we sent next: that is all an agent is

### Now take the safety off


In [ ]:
# Same loop, cut removed. Nothing else changes.
broken = run(honest=False)


**No exception, no warning. Right shape, invented world.**

- `find_book` never ran: put a `print` inside, watch it stay silent
- correct by luck is worse, that is the version you ship


### Two failures that get blamed on the parser

**Both come from the token budget. Neither raises.**


In [ ]:
tiny = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": QUESTION},
    ],
    # Starved on purpose. A reasoning model thinks before it writes a visible
    # character, and the thinking is billed from this same budget.
    max_completion_tokens=64,
)

# Empty. Sixty-four tokens bought hidden reasoning and nothing else.
print("content:      ", repr(tiny.choices[0].message.content))
# `length`, not `stop`, and no exception was raised anywhere.
print("finish_reason:", tiny.choices[0].finish_reason)
# Every completion token spent, with no visible output to show for it.
print("usage:        ", tiny.usage)


In [ ]:
# Same cause, different shape: a reply cut off before the Action line and
# handed straight to the parser.
truncated = "Thought: I should search the cat"

print("content:      ", repr(truncated))
# None again. Identical to the signal a finished answer sends.
print("parse_action: ", repr(parse_action(truncated)))


**The parser is correct. The budget was wrong.**

- both parse to None, and None means finished
- an empty or half-written reply ships as the answer
- `agent/assistant.py` raises a named error instead


## The same request, with a schema

**One parameter. Watch what stops being our problem.**


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            # The name the model prints back to us, and our Python name.
            "name": "find_book",
            # Description and parameter docs are prompt, not documentation:
            # the model's only manual for this tool. Session 3 goes into it.
            "description": "Search the book catalogue. One short keyword works best.",
            # JSON Schema, carrying what the format half of SYSTEM carried.
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "A single keyword"}
                },
                "required": ["query"],
            },
        },
    }
]


In [ ]:
import json

native = client.chat.completions.create(
    model=MODEL,
    # No system prompt at all: the format half is the provider's job now.
    messages=[{"role": "user", "content": QUESTION}],
    # The only addition. Everything else is the request from the first cell.
    tools=TOOLS,
    max_completion_tokens=2000,
)
message = native.choices[0].message

# None: the prose is gone, the call arrived in a field of its own.
print("content:   ", repr(message.content))
print("tool_calls:", message.tool_calls)

# One request, and it stops here on purpose. Sending the result back for a
# second turn 400s on Gemini's OpenAI-compatible endpoint, which a good third of
# this room is on: the reasoning signature does not survive the round trip. A
# different client class fixes it, next session.
for call in message.tool_calls or []:
    # Arguments came over the wire as a string. Text all the way down, only
    # now somebody else is unpacking it.
    arguments = json.loads(call.function.arguments)
    print(f"\ncalling {call.function.name}({arguments}) ->")
    # Running the function never moved. This `for` statement is still ours.
    print(find_book(**arguments))


**The interface did not gain an ability. It gained a schema.**

- `content` is None, the call arrives as an object
- arguments still need `json.loads`: it was a string on the wire
- gone from our code: the format prompt, and `parse_action`
- still ours: running the function
- feeding the result back is the loop from an hour ago; session 2 stops writing it


## Practice

**Your own repository, from scratch.**

- `git init`, a `.env` with the four variables, one Python file
- four TODOs in order, the file runs after each
- you extend this file every session until December

```python
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.environ["LLM_BASE_URL"])
MODEL = os.environ["MODEL_CHEAP"]

# TODO 1. System prompt: name your tool, fix the exact call format, forbid the
# model from writing an Observation. parse_action below must find your format.
SYSTEM = """TODO"""
QUESTION = "TODO"


def my_tool(argument: str) -> str:
    """TODO 2. One string in, one string out. Rename it to what it does and keep
    TOOL pointing at it. The failure case must say what a better argument would
    look like: that sentence is all the model has to work with."""
    raise NotImplementedError


TOOL = my_tool


def parse_action(text: str) -> str | None:
    """TODO 3. Find the call, return the argument, return None when the model
    has finished. Line by line with str.startswith is enough."""
    raise NotImplementedError


def cut_at_observation(text: str) -> str:
    marker = text.find("Observation:")
    return text if marker == -1 else text[:marker].rstrip()


def run(honest: bool = True, steps: int = 5) -> list[str]:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": QUESTION}]
    transcript = [f"Question: {QUESTION}"]
    for _ in range(steps):
        answer = client.chat.completions.create(
            # Pre-set high: hidden reasoning is billed from this budget too.
            model=MODEL, messages=messages, max_completion_tokens=2000
        )
        choice = answer.choices[0]
        reply = choice.message.content or ""
        # Not a parser bug, a budget bug. Say so by name.
        if not reply and choice.finish_reason == "length":
            raise RuntimeError(
                "Empty reply at the length limit: the budget went on hidden "
                "reasoning, not on your parser. Raise max_completion_tokens."
            )
        if honest:
            reply = cut_at_observation(reply)
        transcript.append(reply)
        print(reply)

        argument = parse_action(reply)
        if argument is None:
            break
        observation = f"Observation: {TOOL(argument)}"
        transcript.append(observation)
        print(observation)
        messages.append({"role": "assistant", "content": reply})
        # TODO 4. Hand the observation back: one line. Whose message is it? The
        # model did not write it, and the API will not continue an assistant turn.
    return transcript


if __name__ == "__main__":
    # --break removes the cut. That is the second transcript you commit.
    run(honest="--break" not in sys.argv)
```


**Two rules for the tool.**

- pure Python, no network, no keys: debug your loop, not an API
- a domain no course demo uses: book search, plant care, currency rates, support tickets are taken
- examples list is in the pre-course document


**Required artifact: both transcripts, committed.**

- run it twice, plain and with `--break`
- both into `runs/session-01.md`, tool name and signature on top
- put the cut back before you commit
- one sentence of your own on what the second run shows


**Stretch, if you finish early.**

- a real tool, not a lookup table, with a miss case worth reading
- session 5 is project choice: better to like what you wrote


## Next time

**The loop becomes a graph.**

- the same three moves as nodes and edges, so checkpoints, interrupts and
  tracing can attach without rewriting the loop
- `parse_action` gone, the observation becomes a tool message with an id
- your agent visible node by node in a tracer
